In [ ]:
import os
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

# Ensure local src modules can be imported
sys.path.append(os.path.abspath("."))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
pt_path = r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\final_dataset\labeled_training_dataset.pt"
ckpt_dir = r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)
ckpt_path = os.path.join(ckpt_dir, "two_tower_checkpoint.pt")

print("Loading dataset...")
loaded_dataset = torch.load(pt_path)

print("Loaded successfully!")
print("Number of training pairs:", len(loaded_dataset["label"]))
print("PhD Text Embeddings Matrix Shape:", loaded_dataset["phd_text_emb"].shape)
print("Prof Text Embeddings Matrix Shape:", loaded_dataset["prof_text_emb"].shape)

In [ ]:
class MatchingDataset(Dataset):
    """
    PyTorch Dataset for Two-Tower Applicant-Professor matching.
    """
    def __init__(self, data_dict):
        self.data = data_dict

    def __len__(self):
        return len(self.data["label"])

    def __getitem__(self, idx):
        return {
            "scholar_id": self.data["scholar_id"][idx],
            "prof_id": self.data["prof_id"][idx],
            "label": self.data["label"][idx],
            "similarity_score": self.data["similarity_score"][idx],
            "phd_text_emb": self.data["phd_text_emb"][idx],
            "prof_text_emb": self.data["prof_text_emb"][idx],
            "phd_cat_emb": self.data["phd_cat_emb"][idx],
            "prof_cat_emb": self.data["prof_cat_emb"][idx],
            "phd_num_emb": self.data["phd_num_emb"][idx]
        }

# Create dataset and train/validation split (80/20)
full_dataset = MatchingDataset(loaded_dataset)
val_size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size

generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

In [ ]:
from model.two_tower import TwoTower

model = TwoTower(
    phd_text_dim=384,
    phd_cat_dim=160,
    phd_num_dim=32,
    prof_text_dim=384,
    prof_cat_dim=160,
    output_dim=128,
    dropout=0.2
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [ ]:
class Loss:
    """
    Retrieval Loss functions & Evaluation Metrics for Two-Tower training.
    """
    @staticmethod
    def triplet_margin_loss(q_emb, pos_emb, neg_emb, margin=0.31783701615473536):
        return F.triplet_margin_loss(q_emb, pos_emb, neg_emb, margin=margin)

    @staticmethod
    def batch_triplet_loss(query_embeddings, candidate_embeddings, labels, margin=0.31783701615473536):
        pos_mask = (labels == 1.0)
        neg_mask = (labels == 0.0)
        
        if not pos_mask.any() or not neg_mask.any():
            neg_embs = torch.roll(candidate_embeddings, shifts=1, dims=0)
            return F.triplet_margin_loss(query_embeddings, candidate_embeddings, neg_embs, margin=margin)
            
        pos_queries = query_embeddings[pos_mask]
        pos_candidates = candidate_embeddings[pos_mask]
        neg_candidates = candidate_embeddings[neg_mask]
        
        num_pos = pos_queries.size(0)
        num_neg = neg_candidates.size(0)
        
        indices = torch.randint(0, num_neg, (num_pos,), device=query_embeddings.device)
        sampled_negs = neg_candidates[indices]
        
        return F.triplet_margin_loss(pos_queries, pos_candidates, sampled_negs, margin=margin)

    @staticmethod
    def batch_contrastive_loss(query_embeddings, candidate_embeddings, temperature=0.07, symmetric=True):
        q_norm = F.normalize(query_embeddings, p=2, dim=-1)
        c_norm = F.normalize(candidate_embeddings, p=2, dim=-1)
        
        logits = torch.matmul(q_norm, c_norm.T) / temperature
        batch_size = logits.size(0)
        labels = torch.arange(batch_size, device=logits.device)
        
        forward_loss = F.cross_entropy(logits, labels)
        if not symmetric:
            return forward_loss
            
        backward_loss = F.cross_entropy(logits.T, labels)
        return 0.5 * (forward_loss + backward_loss)

    @staticmethod
    def l2_regularization_loss(model, weight_decay=1e-4):
        l2_reg = 0.0
        for param in model.parameters():
            if param.requires_grad:
                l2_reg += torch.sum(param ** 2)
        return weight_decay * 0.5 * l2_reg

    @staticmethod
    def calculate_retrieval_metrics(query_embeddings, candidate_embeddings):
        q_norm = F.normalize(query_embeddings, p=2, dim=-1)
        c_norm = F.normalize(candidate_embeddings, p=2, dim=-1)
        similarities = torch.matmul(q_norm, c_norm.T).cpu().numpy()
        num_queries = similarities.shape[0]
        positive_ranks = np.empty(num_queries, dtype=np.int32)
        target_indices = np.arange(num_queries)
        for i in range(num_queries):
            ranking = np.argsort(-similarities[i], kind="mergesort")
            positive_ranks[i] = int(np.where(ranking == target_indices[i])[0][0]) + 1
        return {
            "recall_at_10": float(np.mean(positive_ranks <= 10)),
            "recall_at_50": float(np.mean(positive_ranks <= 50)),
            "recall_at_100": float(np.mean(positive_ranks <= 100)),
            "mrr": float(np.mean(1.0 / positive_ranks))
        }

In [ ]:
def train_one_epoch(
    model, dataloader, optimizer, device, 
    loss_mode="combined", margin=0.3178, weight_decay=1e-4, temperature=0.07
):
    model.train()
    total_loss, total_main, total_reg = 0.0, 0.0, 0.0
    for batch in dataloader:
        batch_data = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        optimizer.zero_grad()
        
        query_embs, candidate_embs = model(batch_data)
        labels = batch_data["label"]
        
        if loss_mode == "contrastive":
            main_loss = Loss.batch_contrastive_loss(query_embs, candidate_embs, temperature=temperature)
        elif loss_mode == "triplet":
            main_loss = Loss.batch_triplet_loss(query_embs, candidate_embs, labels=labels, margin=margin)
        elif loss_mode == "combined":
            contrastive = Loss.batch_contrastive_loss(query_embs, candidate_embs, temperature=temperature)
            triplet = Loss.batch_triplet_loss(query_embs, candidate_embs, labels=labels, margin=margin)
            main_loss = 0.5 * contrastive + 0.5 * triplet
        else:
            raise ValueError(f"Unknown loss_mode: {loss_mode}")
            
        reg_loss = Loss.l2_regularization_loss(model, weight_decay=weight_decay)
        loss = main_loss + reg_loss
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_main += main_loss.item()
        total_reg += reg_loss.item()
        
    num_batches = len(dataloader)
    return {
        "loss": total_loss / num_batches,
        "main_loss": total_main / num_batches,
        "reg_loss": total_reg / num_batches
    }

@torch.no_grad()
def evaluate_model(model, dataloader, device):
    model.eval()
    all_query_embs, all_candidate_embs = [], []
    for batch in dataloader:
        batch_data = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        query_embs, candidate_embs = model(batch_data)
        all_query_embs.append(query_embs)
        all_candidate_embs.append(candidate_embs)
    return Loss.calculate_retrieval_metrics(torch.cat(all_query_embs, dim=0), torch.cat(all_candidate_embs, dim=0))

def train_two_tower(
    model, train_loader, optimizer, val_loader=None, epochs=15, 
    loss_mode="combined", margin=0.3178, weight_decay=1e-4, temperature=0.07, 
    device=None
):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)
    
    history = {"train_loss": [], "val_mrr": [], "val_recall_10": [], "val_recall_50": []}
    
    print(f"Starting Two-Tower Training [{loss_mode.upper()} Loss] on [{device}] for {epochs} Epochs...")
    print("=" * 75)
    for epoch in range(1, epochs + 1):
        train_stats = train_one_epoch(
            model=model, 
            dataloader=train_loader, 
            optimizer=optimizer, 
            device=device, 
            loss_mode=loss_mode,
            margin=margin,
            weight_decay=weight_decay, 
            temperature=temperature
        )
        epoch_loss = train_stats["loss"]
        history["train_loss"].append(epoch_loss)
        
        val_str = ""
        if val_loader is not None:
            val_metrics = evaluate_model(model, val_loader, device)
            history["val_mrr"].append(val_metrics["mrr"])
            history["val_recall_10"].append(val_metrics["recall_at_10"])
            history["val_recall_50"].append(val_metrics["recall_at_50"])
            val_str = f" | Val MRR: {val_metrics['mrr']:.4f} | R@10: {val_metrics['recall_at_10']:.4f} | R@50: {val_metrics['recall_at_50']:.4f}"
            scheduler.step(epoch_loss)
                
        print(f"Epoch [{epoch:02d}/{epochs:02d}] Train Loss: {epoch_loss:.4f} (Main: {train_stats['main_loss']:.4f}, Reg: {train_stats['reg_loss']:.4f}){val_str}")
        
    print("=" * 75)
    print("Training Completed!")
    return history

In [ ]:
# Execute Multi-Epoch Training
history = train_two_tower(
    model=model,
    train_loader=train_loader,
    optimizer=optimizer,
    val_loader=val_loader,
    epochs=15,
    loss_mode="combined",  # Options: 'contrastive', 'triplet', 'combined'
    margin=0.3178,
    weight_decay=1e-4,
    temperature=0.07,
    device=device
)

# Save Model State & Optimizer State
checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict()
}
torch.save(checkpoint, ckpt_path)
print(f"Model state and optimizer state saved to: {ckpt_path}")

In [30]:
# Initialize clean model & optimizer
loaded_model = TwoTower(
    phd_text_dim=384,
    phd_cat_dim=160,
    phd_num_dim=32,
    prof_text_dim=384,
    prof_cat_dim=160,
    output_dim=128,
    dropout=0.2
).to(device)

loaded_optimizer = torch.optim.AdamW(loaded_model.parameters(), lr=1e-3)

# Load saved states
checkpoint = torch.load(ckpt_path, map_location=device)
loaded_model.load_state_dict(checkpoint["model_state_dict"])
loaded_optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
loaded_model.eval()

print(f"Successfully loaded model_state_dict and optimizer_state_dict from: {ckpt_path}")

Successfully loaded model_state_dict and optimizer_state_dict from: C:\Users\ps302\OneDrive\Desktop\Recommend\src\checkpoints\two_tower_checkpoint.pt


In [31]:
with torch.no_grad():
    loaded_model.eval()
    
    # Compute Professor Candidate Embeddings
    prof_text = loaded_dataset["prof_text_emb"].to(device)
    prof_cat = loaded_dataset["prof_cat_emb"].to(device)
    prof_embeddings = loaded_model.candidate_tower(prof_text, prof_cat)  # Shape: (N, 128)
    
    # Extract Sample PhD Query Embeddings
    phd_text = loaded_dataset["phd_text_emb"][0].unsqueeze(0).to(device)
    phd_cat = loaded_dataset["phd_cat_emb"][0].unsqueeze(0).to(device)
    phd_num = loaded_dataset["phd_num_emb"][0].unsqueeze(0).to(device)
    query_embedding = loaded_model.query_tower(phd_text, phd_cat, phd_num)  # Shape: (1, 128)
    
    # Compute Cosine Similarities & Get Top-10 Matches
    similarities = torch.matmul(query_embedding, prof_embeddings.T).squeeze(0)
    top_k_scores, top_k_indices = torch.topk(similarities, k=10)
    
    # Display Recommendations
    scholar_id = loaded_dataset["scholar_id"][0].item()
    prof_ids = loaded_dataset["prof_id"]
    
    print("\n===== Top-10 Recommended Professors =====")
    print(f"Scholar ID: {scholar_id}")
    for rank, (idx, score) in enumerate(zip(top_k_indices.cpu().tolist(), top_k_scores.cpu().tolist()), start=1):
        prof_id = prof_ids[idx].item() if isinstance(prof_ids, torch.Tensor) else prof_ids[idx]
        print(f"Rank {rank:02d}: Professor ID {prof_id} | Similarity Score: {score:.4f}")


===== Top-10 Recommended Professors =====
Scholar ID: 150
Rank 01: Professor ID 8326 | Similarity Score: 0.8451
Rank 02: Professor ID 147 | Similarity Score: 0.8036
Rank 03: Professor ID 147 | Similarity Score: 0.8036
Rank 04: Professor ID 1901 | Similarity Score: 0.7833
Rank 05: Professor ID 4382 | Similarity Score: 0.7662
Rank 06: Professor ID 4382 | Similarity Score: 0.7662
Rank 07: Professor ID 4382 | Similarity Score: 0.7662
Rank 08: Professor ID 8209 | Similarity Score: 0.7586
Rank 09: Professor ID 5689 | Similarity Score: 0.6822
Rank 10: Professor ID 5689 | Similarity Score: 0.6822


In [33]:
import json
import pandas as pd

def get_recommendation_details(
    scholar_id,
    prof_ids,
    phd_json_path=r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\processed\phd\sop\gpt_extracted_data.json",
    prof_csv_path=r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\processed\prof\extracted_04_prof_data.csv"
):
    """
    Retrieves PhD Applicant profile details and matching Professor profiles for recommended Professor IDs.
    """
    # 1. Load PhD Scholar Data from JSON
    with open(phd_json_path, 'r', encoding='utf-8') as f:
        phd_data = json.load(f)
        
    phd_map = {item['scholar_id']: item for item in phd_data if 'scholar_id' in item}
    scholar_info = phd_map.get(scholar_id, {})
    
    # 2. Load Professor Data from CSV
    prof_df = pd.read_csv(prof_csv_path)
    
    # Deduplicate prof_ids while preserving rank order
    unique_prof_ids = list(dict.fromkeys(prof_ids))
    
    # Filter matching professor rows
    matched_profs = prof_df[prof_df['id'].isin(unique_prof_ids)].copy()
    
    # Assign rank order matching recommendation list
    matched_profs['rank'] = matched_profs['id'].map(lambda x: unique_prof_ids.index(x) + 1)
    matched_profs = matched_profs.sort_values('rank').reset_index(drop=True)
    
    # 3. Print Summary Output
    print("==================================================")
    print(f"🎓 SCHOLAR PROFILE [ID: {scholar_id}]")
    print(f"Name: {scholar_info.get('name', 'N/A')}")
    print(f"University: {scholar_info.get('university', 'N/A')}")
    print(f"Department: {scholar_info.get('department', 'N/A')}")
    print(f"Research Interests: {scholar_info.get('research_interests', 'N/A')}")
    print("==================================================")
    print("👨‍🏫 TOP RECOMMENDED PROFESSORS:")
    print("==================================================")
    
    display_cols = [col for col in ['rank', 'id', 'name', 'department', 'title', 'expertise'] if col in matched_profs.columns]
    print(matched_profs[display_cols].to_string(index=False))
    
    return scholar_info, matched_profs


# ==========================================
# Example Usage:
# ==========================================
recommended_prof_ids = [8326, 147, 1901, 4382, 8209, 5689]
scholar_info, prof_details_df = get_recommendation_details(
    scholar_id=150, 
    prof_ids=recommended_prof_ids
)


🎓 SCHOLAR PROFILE [ID: 150]
Name: Maya L. Patel
University: Stanford University
Department: Electrical and Computer Engineering
Research Interests: ['Autonomous navigation for aerial robotics', 'Multi‑agent reinforcement learning', 'Safety‑critical AI for UAV swarms', 'Sensor fusion and visual‑inertial odometry', 'Edge AI and low‑power inference', 'Explainable decision‑making in autonomous systems', 'Human‑robot interaction in mixed‑initiative missions']
👨‍🏫 TOP RECOMMENDED PROFESSORS:
 rank   id                           name                                     department                                                                                                                        title                                                                                                                                                                                                                                                                                                        